In [5]:
# Ensure we are running from the project root so all relative paths work
import os
from gbheatmodel.path import PROJECT_ROOT

os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)

Project root: C:\Users\s2583817\OneDrive - University of Edinburgh\Documents\Projects\PhD_Project\Python\GBHeatModel


In [6]:
import pandas as pd
import matplotlib.pyplot as plt

from gbheatmodel import run_heat_demand_pipeline

In [7]:
# 🔧 User inputs for heating demand pipeline

weather_year = 2010        # weather data / patterns year
future_year = 2048         # target future year
FES_year = "2022"          # FES dataset version
FES_scenario = "LW"        # "LW", "CT", "ST", "FS", ...

# Optional: a grid to inspect later
grid_id = "-3.5_56.0"

In [10]:
results = run_heat_demand_pipeline(
    weather_year=weather_year,
    future_year=future_year,
    FES_year=FES_year,
    scenario=FES_scenario,
)

LA_overlaps     = results["LA_overlaps"]
df_merged      = results["df_merged"]
total_dwellings = results["total_dwellings"]
grid_square_hd  = results["grid_square_hd"]
profiles        = results["profiles"]

print("✅ Heating demand pipeline finished.")
print(f"- LA overlaps rows        : {len(LA_overlaps)}")
print(f"- Merged FES rows         : {len(df_merged)}")
print(f"- Total dwellings rows    : {len(total_dwellings)}")
print(f"- Grid-square HD rows     : {len(grid_square_hd)}")
print(f"- Number of grid profiles : {len(profiles)}")

Sample FES LAs: ['Aberdeen City', 'Aberdeenshire', 'Adur', 'Allerdale', 'Amber Valley', 'Angus', 'Argyll And Bute', 'Arun', 'Ashfield', 'Ashford']
Sample Grid LAs: ['Isles Of Scilly' 'Cornwall' 'Plymouth' 'South Hams' 'Torbay'
 'West Devon' 'Teignbridge' 'East Devon' 'Dorset' 'Isle Of Wight']
✅ All heating profiles already exist. Loading from disk.


KeyError: 'df_merged'

In [ ]:
# Quick check: total dwellings across all grids and technologies
total_dwellings_total = total_dwellings.drop(columns=["Grid_ID"], errors="ignore").sum().sum()
print("Total dwellings across GB:", total_dwellings_total)

In [ ]:
# 🔍 Inspect and plot one grid's total half-hourly heat profile

if grid_id not in profiles:
    raise ValueError(f"Grid ID {grid_id} not found in profiles. "
                     f"Available examples: {list(profiles.keys())[:5]}")

df = profiles[grid_id]  # columns = technologies

# Sum across technologies to get total heat demand
series_total = df.sum(axis=1)

# Compute average daily profile (48 half-hours)
daily_matrix = series_total.values.reshape(-1, 48)
daily_avg = daily_matrix.mean(axis=0)

time_axis = [f"{int(i/2):02d}:{'30' if i % 2 else '00'}" for i in range(48)]

plt.figure(figsize=(10, 4))
plt.plot(time_axis, daily_avg)
plt.xticks(rotation=90)
plt.title(f"Average daily heat profile for grid {grid_id} "
          f"({weather_year} weather, {future_year}, {fes_year}/{fes_scenario})")
plt.ylabel("Heat demand (units)")
plt.tight_layout()
plt.show()
